# Hyporheic FloPy Main Workflow
Combines preprocessing, model setup, execution, and post-processing into a single notebook.

## Preprocessing

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import geopandas as gpd

from VQuintana.functions import raster_utils as ru
from VQuintana.functions import model_utils as mu
from VQuintana.functions import main_utils as mutils

# User-updatable paths
project_dir = Path("VQuintana/CH00365")
data_dir = Path("VQuintana/notebooks")
output_dir = data_dir / "Hyporheic_output"

terrain_raster = data_dir / "reprojected_terrain_raster.tif"
water_surface_raster = data_dir / "reprojected_water_surface_elevation_raster.tif"
domain_shp = project_dir / "GWDomain.shp"
left_boundary_shp = project_dir / "L_FPL.shp"
right_boundary_shp = project_dir / "R_FPL.shp"
river_cells_csv = data_dir / "river_cells.csv"

# Load rasters
terrain, transform, crs, nodata, bounds_box = ru.load_raster(terrain_raster)
terrain = ru.interpolate_na(ru.mask_nodata(terrain, nodata))
water_surface, _, _, _, _ = ru.load_raster(water_surface_raster)
water_surface = ru.interpolate_na(ru.mask_nodata(water_surface, nodata))


## Initialization

In [ ]:
cfg = SimpleNamespace(
    sim_name="hyporheic_sim",
    gwf_name="gwf_model",
    gwf_ws=output_dir,
    mp7_ws=output_dir / "mp7",
    md6_exe_path="mf6",
    mp7_exe_path="mp7",
    cell_size_x=100.0,
    cell_size_y=100.0,
    nlay=1,
    nper=1,
    nstp=1,
    perlen=1.0,
    tsmult=1.0,
    time_units="DAYS",
    bed_elevation=float(np.nanmin(terrain)),
    kh=1.0,
    kv=1.0,
    headfile="head.hds",
    budgetfile="budget.cbc",
)

xmin, ymin, xmax, ymax = ru.raster_extent(transform, terrain.shape[1], terrain.shape[0])
cfg.xmin, cfg.ymin, cfg.xmax, cfg.ymax = xmin, ymin, xmax, ymax
width_ft = xmax - xmin
height_ft = ymax - ymin
cfg.ncol, cfg.nrow = ru.grid_dimensions(width_ft, height_ft, cfg.cell_size_x, cfg.cell_size_y)
grid_x, grid_y = ru.generate_grid_centres(cfg.ncol, cfg.nrow, cfg.cell_size_x, cfg.cell_size_y, xmin, ymin)
grid_points = ru.grid_to_geodataframe(grid_x, grid_y, crs)

cfg.tops = [terrain]
cfg.botm = [terrain - cfg.bed_elevation]
cfg.raster_crs = crs


## Model Domain

In [ ]:
domain_gdf = gpd.read_file(domain_shp)
grid_polys = mu.build_grid_polygons(grid_x, grid_y, cfg.cell_size_x, cfg.cell_size_y, crs)
idomain = mu.idomain_from_domain(grid_polys, domain_gdf, cfg.nlay, cfg.nrow, cfg.ncol)


## Define Boundary

In [ ]:
left_boundary = gpd.read_file(left_boundary_shp)
right_boundary = gpd.read_file(right_boundary_shp)
upstream, downstream = mu.make_up_down_stream(left_boundary, right_boundary, crs)
river_df = pd.read_csv(river_cells_csv)
river_cells = [tuple(map(int, r)) for r in river_df[['k','i','j']].values]


## Boundary Conditions

In [ ]:
boundary_cells = [(0, i, j) for (_, i, j) in river_cells]
chd_data = mutils.interpolate_gw_elevation_first_layer_only(
    boundary_cells,
    head_first=float(np.min(water_surface)),
    head_last=float(np.max(water_surface)),
)


## Run Models

In [ ]:
sim, gwf = mutils.build_gwf_model(cfg, chd_data, idomain)
mp_fwd, mp_bwd = mutils.build_particle_models(cfg.sim_name, gwf, river_cells, mp7_ws=cfg.mp7_ws, exe_path=cfg.mp7_exe_path)
mutils.write_models(sim, mp_fwd, mp_bwd)
mutils.run_models(sim, mp_fwd, mp_bwd)


## Particle Results

In [ ]:
# Placeholder for particle tracking results
# mutils.process_and_export_modpath7_results(mp_fwd, mp_bwd)


## Tech Note Figures

In [ ]:
# Placeholder for figure generation
